# Predicting Review Helpfulness Before the First Vote

### A leakage-free study of Amazon Reviews 2023 using PySpark

**CSE 4262 — Data Analytics Lab** · Gr-03 / Gr-06

---

When a review is posted on Amazon it has **zero helpful votes**, yet the platform must decide
immediately where to place it. Can that placement be predicted from the review itself?

This notebook runs the full study. Its central concern is a validity problem that affects much
of the existing literature:

> Helpful votes accumulate with **exposure**. A 2015 review has had eight years to collect
> clicks; a 2023 review has had weeks. A model given any time-correlated feature learns to
> detect *review age* and scores well without learning anything about *review quality*.

A recent study on this same dataset ([arXiv:2412.02884](https://arxiv.org/abs/2412.02884))
reported **96.91% accuracy** from a reviewer-history feature that, for single-review reviewers,
is the prediction target restated.

We eliminate that class of error structurally, measure how much of the prior performance it
accounted for, and re-evaluate the task honestly.


## 1. Environment

In [ ]:
# Colab needs PySpark installed; Kaggle already has it.
try:
    import pyspark
    print("PySpark already present:", pyspark.__version__)
except ImportError:
    !pip install -q pyspark
    import pyspark
    print("PySpark installed:", pyspark.__version__)

try:
    import statsmodels
    print("statsmodels:", statsmodels.__version__)
except ImportError:
    !pip install -q statsmodels
    import statsmodels
    print("statsmodels installed:", statsmodels.__version__)

## 2. Data

Download the extract from Kaggle, or set `DATA_PATH` yourself.

In [ ]:
import os, glob

DATA_PATH = os.environ.get("AMAZON_DATA_PATH")

if not DATA_PATH or not os.path.exists(DATA_PATH):
    try:
        import kagglehub
    except ImportError:
        !pip install -q kagglehub
        import kagglehub

    path = kagglehub.dataset_download(
        "tahsintajware/amazon-reviews-2023-three-category-extract")
    hits = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
    DATA_PATH = hits[0] if hits else path

os.environ["AMAZON_DATA_PATH"] = DATA_PATH
print("Data:", DATA_PATH)
print("Size: %.1f MB" % (os.path.getsize(DATA_PATH) / 1e6))

## 3. Inspect the raw data

Before any modelling, establish the basic shape of the corpus — especially the **base rate**,
which determines which evaluation metrics are meaningful.

In [ ]:
import pandas as pd

df_peek = pd.read_csv(DATA_PATH, low_memory=False)

print(f"TOTAL ROWS: {len(df_peek):,}")
print("COLUMNS   :", list(df_peek.columns))

print("\nROWS PER CATEGORY")
print(df_peek["Category"].value_counts().to_string())

print("\nHELPFUL VOTES")
print(f"  mean          {df_peek['helpful_vote'].mean():.3f}")
print(f"  max           {df_peek['helpful_vote'].max()}")
print(f"  pct with >=1  {100 * (df_peek['helpful_vote'] >= 1).mean():.2f}%   <- THE BASE RATE")
print(f"  pct with >=10 {100 * (df_peek['helpful_vote'] >= 10).mean():.2f}%")

ts = pd.to_datetime(df_peek["timestamp"], unit="ms", errors="coerce")
print(f"\nDATE RANGE: {ts.min()}  ->  {ts.max()}")
print("\nROWS PER YEAR")
print(ts.dt.year.value_counts().sort_index().to_string())

**Read the base rate carefully.** If it is around 10%, then a model that predicts
*"never helpful"* for every single review achieves roughly **90% accuracy** while being
completely useless.

This is why accuracy is not reported anywhere in this study. Primary metric is **PR-AUC**,
reported alongside **PR-AUC lift** (PR-AUC ÷ base rate), which makes a model that learned
almost nothing impossible to disguise.

### The exposure confound, visible

Plot average helpful votes against review year. If the line rises steeply with age, the target
is measuring exposure rather than quality — and that is what the pipeline must correct.

In [ ]:
import matplotlib.pyplot as plt

peek = df_peek.assign(year=ts.dt.year)
by_year = peek.groupby("year").agg(
    avg_votes=("helpful_vote", "mean"),
    pct_any=("helpful_vote", lambda s: 100 * (s >= 1).mean()),
    n=("helpful_vote", "size"),
).reset_index()
by_year = by_year[by_year["n"] >= 100]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(by_year["year"], by_year["avg_votes"], marker="o", color="#db2777")
ax[0].set_title("Average helpful votes by review year")
ax[0].set_xlabel("Year"); ax[0].set_ylabel("Mean helpful votes")
ax[1].plot(by_year["year"], by_year["pct_any"], marker="o", color="#2563eb")
ax[1].set_title("% of reviews receiving any helpful vote")
ax[1].set_xlabel("Year"); ax[1].set_ylabel("Percent")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Older reviews score higher because they have been visible longer,")
print("not because they are better written. This is the confound the")
print("pipeline controls for by censoring and conditioning on exposure.")

## 4. Load the pipeline

The analysis code lives in `src/`. Clone the repository, or upload the `src/` folder alongside
this notebook.

In [ ]:
import sys

SRC = None
for candidate in ["src", "../src", "/kaggle/working/src", "/content/src"]:
    if os.path.isdir(candidate):
        SRC = os.path.abspath(candidate)
        break

if SRC is None:
    raise FileNotFoundError(
        "Could not find src/. Clone the repo or upload the src/ folder next to this notebook."
    )

sys.path.insert(0, SRC)
print("Pipeline:", SRC)

import config
print("\nKey methodological settings")
print(f"  collection date      {config.COLLECTION_DATE}")
print(f"  min exposure days    {config.MIN_EXPOSURE_DAYS}")
print(f"  train / val quantile {config.TRAIN_QUANTILE} / {config.VAL_QUANTILE}")
print(f"  helpful threshold    >= {config.HELPFUL_THRESHOLD} vote(s)")

## 5. Run the study

This executes all five experiments and writes every table and figure.

| Experiment | Question |
| --- | --- |
| 1 — Leakage decomposition | How much of prior performance was artefact? |
| 2 — Feature ablation | Which feature blocks actually contribute? |
| 3 — Model comparison | How well can this be done honestly? |
| 4 — Product-type moderation | Do determinants differ for search vs experience goods? |
| 5 — Exposure bias | How unequally are helpful votes distributed? |

Runtime scales with corpus size — roughly 8 minutes per 60k rows on a free Colab CPU.

In [ ]:
from run_analysis import main

RESULTS = main(run_text_features=True, run_interaction=True)

## 6. The headline result

Four conditions, differing only in the two design choices under study. The gap between the
first and last row is the size of the methodological artefact.

In [ ]:
leak = RESULTS.get("tbl04_leakage_decomposition")
if leak is not None:
    display(leak)
    if len(leak) >= 4:
        a, d = leak.iloc[0]["pr_auc"], leak.iloc[3]["pr_auc"]
        c = leak.iloc[2]["pr_auc"]
        b = leak.iloc[1]["pr_auc"]
        print(f"\nTotal inflation      {a - d:+.4f} PR-AUC  ({a:.4f} -> {d:.4f})")
        print(f"  from leaky feature {a - c:+.4f}")
        print(f"  from random split  {a - b:+.4f}")

In [ ]:
from IPython.display import Image, display
import os

for fig in ["fig01_leakage_decomposition", "fig02_feature_ablation",
            "fig03_model_comparison", "fig04_product_type_moderation",
            "fig05_fairness"]:
    p = os.path.join(config.FIG_DIR, fig + ".png")
    if os.path.exists(p):
        display(Image(filename=p))

## 7. Models against baselines

Trivial baselines are shown alongside the models. Omitting them is how a modest result gets
presented as a strong one: if review length alone reaches most of a model's score, the model
has added very little.

In [ ]:
models = RESULTS.get("tbl06_model_comparison")
if models is not None:
    display(models)

## 8. Does product type moderate helpfulness?

Nelson (1970) and Mudambi & Schuff (2010) predict that helpfulness is a property of the
**review–product pair**, not the review alone:

- **H1** — depth raises helpfulness *more* for search goods (phones), whose quality is
  verifiable from published specifications.
- **H2** — extremity is *penalised* for search goods, where an extreme rating signals an
  idiosyncratic reviewer rather than product quality.

The original test used 1,587 reviews of six products. This tests at corpus scale.

In [ ]:
for name in ["tbl09_construct_comparison", "tbl10_interaction_model",
             "tbl11_marginal_effects"]:
    t = RESULTS.get(name)
    if t is not None:
        print(f"\n--- {name} ---")
        display(t)

## 9. Exposure bias

How unequally are helpful votes distributed, and does arriving late to a product page cost a
review attention it deserved?

In [ ]:
for name in ["tbl13_exposure_bias", "tbl14_vote_concentration"]:
    t = RESULTS.get(name)
    if t is not None:
        print(f"\n--- {name} ---")
        display(t)

## 10. Generate the report

`RESULTS.md` is written directly from the tables above, so the prose and the numbers cannot
drift apart.

In [ ]:
!python {SRC}/make_report.py

## 11. Demonstration

Score a draft review and explain the estimate.

In [ ]:
sys.path.insert(0, os.path.dirname(SRC))
from review_scorer import extract_features, heuristic_score, generate_feedback, render

draft = ("I bought this case for my iPhone 14 Pro about three months ago. The drop "
         "protection is genuinely good - it survived a fall onto concrete from waist "
         "height with no damage. The raised corner lips are what actually do the work. "
         "Downsides: the button covers are stiff and it picks up lint in pockets. At "
         "25 dollars it is reasonable but not exceptional.")

feats = extract_features(draft, title="Good protection, stiff buttons",
                         rating=4, verified=True, has_images=False)
render(heuristic_score(feats), feats,
       generate_feedback(feats, "Cell Phones", "search"), "Cell Phones")

---

## Summary

1. **Helpful votes measure exposure as much as quality.** Any model given a time-correlated
   feature will exploit that and report inflated performance.
2. **The inflation is measurable.** Experiment 1 separates it into the part caused by random
   splitting and the part caused by lookahead in reviewer features.
3. **Honest performance on this task is modest.** Reporting that plainly, with trivial
   baselines alongside, is the contribution — not a shortcoming.
4. **What makes a review helpful is not universal.** It depends on the kind of product.
5. **The vote mechanism is structurally unfair to new reviews**, and a ranker trained naively
   on it will reproduce that unfairness.

Full write-up in `REPORT.md`; all generated results in `RESULTS.md`.